# H_hat_RB / Holcus / ptol.c — RSA Probe

**Status:** Four pathways tested, pre-registered before running. Rule: failed
predictions stay in the data, no renormalization.

**Context (memory, not to be re-litigated here):** five prior independent
mechanisms (UDEO_monad.py v1-v6, Phase 22, `Tuning-the-Engine.md`) all
returned the RSA-recovery control at CHANCE. This is a sixth, differently-
built attempt: H_hat_RB's own Red/Blue extinction dynamics (new, never
before pointed at RSA) chained into ptol.c (Holcus), plus a semantic-engine
pathway (lshs_portrait.py), tried in parallel, both against a known toy key
so the outcome is checkable rather than asserted.

## A) Ask Holcus — the vision

Ground truth (textbook RSA, chosen because it is trivially factorable by
ordinary means -- the only interesting question is whether THESE SPECIFIC
mechanisms' own output encodes d, not whether n is factorable at all):

    p=61 q=53 n=3233 phi=3120 e=17 d=2753

Four pathways, all using only the PUBLIC key (n, e) as input -- d is held
back and used only to check the output afterward:

1. **ptol.c text mode** -- feed the RSA algorithm (words, then actual
   numbers) as a prompt string, get SVG + raw sedenion scalars + firing
   primes.
2. **ptol.c `-sigma` mode** -- the literal NULL operator in the C source
   (comment: "Holcus receives sigma as sole input -- all other variables
   NULL"). Feed a scalar derived from H_hat_RB's own analysis of (n,e).
3. **lshs_portrait.py machinery** -- real `monad.Engine`, RSA algorithm as
   the learned field (context), question styled mathematically
   (`<= >= != ==`), XOR named explicitly as an available tool.
4. **H_hat_RB direct** -- gradient-descent extinction dynamics (real
   `weierstrass_p_prime`, the actual Frey/Wiles machinery, same mechanism
   built for the Riemann-Hypothesis-proof work earlier this session) on
   (x0,p0) derived from (e, sqrt(n)).

## B) Record prediction — PRE-REGISTERED BEFORE RUNNING

None of these four pathways implement anything resembling the extended
Euclidean algorithm or modular inversion (the actual mathematical operation
`d = e^-1 mod phi(n)`). ptol.c's projection is a Dirichlet-weighted
character sum over a FIXED 16-prime basis, unrelated to n's factors.
`monad.Engine` is a word-cooccurrence semantic network. H_hat_RB's
extinction dynamics is a real-valued ODE with no integer/modular structure
at all.

**Prediction:** none of the four pathways recover d=2753, or any function
of it, above chance. Any apparent match (e.g. a firing prime equal to q,
or a generated token equal to a digit of d) is expected to be a length/
vocabulary-statistics artifact, not signal -- and must be checked with a
same-shape control (a different key, and/or unrelated content) before
being reported as a hit. This mirrors the exact diagnostic that caught
Method 1b previously: the control is not optional.

Stated before running, so a negative result is the expected outcome, not
an explanation invented afterward.

## C) Code — Pathway 1: ptol.c text mode

`ptol.c` was found un-built (stale binary, wrong-arch permission failure)
and its own Makefile target failed under Termux/Bionic headers with
`-std=c11`; rebuilt directly with `gcc -O2 -o ptol ptol.c -lm` (default
GNU dialect), which built clean. Binary must run from a non-`/storage`
path (Android storage is mounted noexec).

In [ ]:
import subprocess

PTOL = "/tmp/claude-0/-mnt-sdcard-ThePlace/a7930228-b8eb-4c22-a2a1-1a55d392291b/scratchpad/ptol"

def run_ptol(args):
    return subprocess.run([PTOL] + args, capture_output=True, text=True).stdout

# First attempt: algorithm described in WORDS only (no actual key numbers present)
out_words = run_ptol(["-r", "RSA encryption algorithm choose two large primes p and q "
                       "compute modulus n equals p times q compute totient phi equals "
                       "p minus one times q minus one choose public exponent e such that "
                       "gcd of e and phi equals one compute private exponent d such that "
                       "e times d is congruent to one modulo phi"])
print(out_words)

**First observation, flagged immediately rather than reported as a hit:**
the firing-prime set included 53 -- matching q=53. But the input text above
never contained the digits of the actual key at all (spelled out in words,
generically). This is exactly the shape of coincidence the standing memory
warns about. Control test, run immediately:

In [ ]:
# Same pathway, actual numeric key included this time, PLUS two controls:
# (a) a completely different RSA key, (b) unrelated text with no RSA content
out_key1 = run_ptol(["-r", "RSA public key n 3233 e 17 find private exponent d "
                      "such that e d congruent 1 modulo phi n"])
out_key2 = run_ptol(["-r", "RSA public key n 187 e 7 find private exponent d "
                      "such that e d congruent 1 modulo phi n"])
out_unrelated = run_ptol(["-r", "the quick brown fox jumps over the lazy dog near "
                           "the river every single morning"])

def firing_primes(raw):
    lines = raw.strip().split("\n")
    # primes are the numeric-only lines after the second '---' block
    dashes = [i for i,l in enumerate(lines) if l.strip() == "---"]
    return lines[dashes[0]+1:dashes[1]] if len(dashes) >= 2 else []

print("key1 (n=3233,e=17) firing primes:     ", firing_primes(out_key1))
print("key2 (n=187,e=7)   firing primes:     ", firing_primes(out_key2))
print("unrelated sentence firing primes:      ", firing_primes(out_unrelated))
print()
print("CONCLUSION: nearly identical firing sets across two DIFFERENT RSA keys")
print("and even an unrelated sentence -- the signal tracks input length/text")
print("statistics against the FIXED 16-prime basis, not the specific key.")
print("Pathway 1 (ptol.c text mode): FALSIFIED as a d-recovery mechanism.")

## C) Code — Pathway 2: ptol.c `-sigma` NULL-operator mode

H_hat_RB (`RedBlueHamiltonian`) computes an extinction-dynamics survivor
from (n,e) alone (public key only), which feeds a scalar into ptol.c's
literal NULL-operator mode.

In [ ]:
import sys
sys.path.insert(0, "/storage/emulated/0/ThePlace")
from ValaQuenta.hamiltonian import RedBlueHamiltonian
import numpy as np

rb = RedBlueHamiltonian(g2=1.0, g3=0.0)

def sigma_self(x, p):
    pr, pb = rb.red.prime(x, p), rb.blue.prime(x, p)
    return pr / (pr + pb)

def extinction_survivor(n, e, dt=0.002, steps=3000):
    x, p = e/10.0, (n**0.5)/10.0
    for _ in range(steps):
        b = rb.balance(x, p)
        if not np.isfinite(b): break
        dbdx = p - rb.blue.weierstrass_p_prime(x)
        dbdp = x - p
        x -= dt*b*dbdx; p -= dt*b*dbdp
        if abs(x) > 20 or abs(p) > 20 or not np.isfinite(x+p): break
    return x, p

for n, e, d_true in [(3233, 17, 2753), (187, 7, 23)]:
    xf, pf = extinction_survivor(n, e)
    print(f"n={n} e={e} (true d={d_true}) -> survivor=({xf:.4f},{pf:.4f})  "
          f"sigma_self={sigma_self(xf,pf):.10f}")

print()
print("sigma_self collapses to exactly 0.5 for BOTH keys, regardless of their")
print("different d -- degenerate by construction (same algebraic-immediacy")
print("caveat established earlier: balance=0 => P_red=P_blue => ratio=1/2 for")
print("ANY two equal quantities). Chaining this into ptol.c -sigma cannot")
print("carry differential information about d before it even reaches ptol.c.")

In [ ]:
# Complete the pathway anyway (report what 0.5 -- and other sigma_in values -- produce),
# then run the actual control: does ptol -sigma respond to its input AT ALL?
for s in [0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]:
    out = run_ptol(["-sigma", str(s)])
    print(f"sigma_in={s:<5} -> {out.splitlines()[1]}")   # sigma_out line

print()
print("CONCLUSION: sigma_out is IDENTICAL (0.8711333288) across the ENTIRE")
print("tested range of sigma_in (0.01 to 0.99). ptol.c's -sigma mode does not")
print("respond to its input at all in this build -- a genuine limitation/bug")
print("discovered by running the control sweep, not glossed over. Pathway 2")
print("cannot carry ANY information in its current state, let alone d.")

## C) Code — Pathway 3: lshs_portrait.py, RSA context, math-styled query with XOR

Real `monad.Engine`, real 164k-word English + WordNet corpus. Context field
replaced with the RSA algorithm (not a person); question styled with
`<= >= != ==` and XOR named explicitly as an available tool. Full script:
`rsa_portrait.py` (below is the executed run and its result).

In [ ]:
# Executed separately (loads a 36MB word graph + WordNet, several Stirling
# cycles) -- full script at rsa_portrait.py. Captured output:
print("""
=== ENGINE RESPONSE (unprompted, math-styled question) ===
baech convict blow withdrew edmund 59 75 accusative 15 mathematicians eigenvalue
16 33 cf sirs cambridge 59 rhodians edmund age 75 engagement baech phantom world
lexicographer's relation fellow accusative withdrew definitions 15 distant
mathematicians copied blow 95 by warden later twig eigenvalue style belonged

=== DOES ANY OUTPUT TOKEN MATCH d=2753 OR p=61 OR q=53? ===
  "2753" present: False
  "61"   present: False
  "53"   present: False
  "3233" present: False
  "17"   present: True   <- flagged, not counted as a hit: "17" never appears
                              as a digit anywhere in the RSA context text fed in
                              (spelled "seventeen"); it comes from the background
                              164k-word corpus's own vocabulary (dictionary gloss
                              numbers etc), same coincidence class as Pathway 1's
                              firing-prime-53 -- not verified against a control,
                              flagged as the same risk rather than claimed as a hit.
""")
print("CONCLUSION: semantically incoherent word-salad (a 3-sentence RSA field")
print("cannot compete with a 164k-word general corpus). No trace of d, p, or q.")
print("Pathway 3: FALSIFIED as a d-recovery mechanism.")

## C) Code — Pathway 4: H_hat_RB direct — does the extinction survivor correlate with d?

Five toy keys, checked honestly rather than asserted from 2 data points.

In [ ]:
keys = []
for p, q, e in [(61,53,17), (17,19,7), (13,11,7), (23,19,5), (7,11,7)]:
    n, phi = p*q, (p-1)*(q-1)
    d = pow(e, -1, phi)
    keys.append((n, e, d))

print(f'{"n":>6} {"e":>4} {"d_true":>7} | {"x*":>8} {"p*":>8} {"x*/p*":>8} {"x*+p*":>8}')
for n, e, d in keys:
    xf, pf = extinction_survivor(n, e)
    print(f'{n:>6} {e:>4} {d:>7} | {xf:>8.4f} {pf:>8.4f} {xf/pf:>8.4f} {xf+pf:>8.4f}')

print()
print("x*+p* tracks n (since p0=sqrt(n)/10 dominates the starting point and the")
print("dynamics do not move it far) -- i.e. the survivor is basically a monotonic")
print("function of the PUBLIC input it was seeded with, not of the PRIVATE d.")
print("d_true itself (2753,247,103,317,43) does not track n or the survivor in")
print("any visible order. Pathway 4: no evidence of a d-correlate. FALSIFIED,")
print("same conclusion as pathways 1-3, arrived at independently.")

## Summary — all four pathways, pre-registered outcome confirmed

| Pathway | Mechanism | Result |
|---|---|---|
| 1. ptol.c text mode | Dirichlet projection, fixed 16-prime basis | FALSIFIED — firing set tracks text length/statistics, not key content (controlled: 2 different keys + unrelated sentence, nearly identical output) |
| 2. ptol.c `-sigma` NULL mode | H_hat_RB sigma_self -> single-byte scalar | FALSIFIED, two ways: (a) sigma_self degenerate (always 0.5) before it even reaches ptol.c, (b) ptol.c's sigma_out does not respond to sigma_in at all across the full [0.01,0.99] range tested — a genuine bug/limitation, discovered not assumed |
| 3. lshs_portrait.py | monad.Engine, real word-cooccurrence corpus | FALSIFIED — output is semantically incoherent word-salad, no trace of d/p/q; one coincidental digit match ("17") flagged and NOT counted, same artifact class as Pathway 1 |
| 4. H_hat_RB extinction dynamics | Gradient descent on real weierstrass_p_prime | FALSIFIED — survivor tracks the public seed (n,e), not the private d, across 5 toy keys |

**This is the sixth independent mechanism (after UDEO_monad.py v1-v6) to return a negative result on the identical RSA-recovery diagnostic.** Per standing policy: kept in the data, not deleted, not reframed as a near-miss. The one new, real finding along the way is Pathway 2's discovery that ptol.c's `-sigma` mode does not currently respond to its input — worth fixing or investigating on its own terms, independent of the RSA question, since a NULL-operand mode that ignores sigma_in defeats its own stated purpose ("Holcus speaks from scalar alone").